In [ ]:
# IMPORTS
from memo import memo, domain
from math import exp
import jax
import numpy as np
import jax.numpy as jnp
from jax.scipy.stats import beta as jax_beta
import matplotlib.pyplot as plt

# GLOBALS
INFO, ACTION = 0, 1
CAN, IMP, NULL = 0, 1, 2
ACT, PASS = 0, 1
goals = [INFO, ACTION]
U = [CAN, IMP, NULL]
R = [ACT, PASS]

# COSTS
c_can, c_imp = .1, .7 # utterance costs for speaker

mu_vals = np.linspace(0.001, .999, 50)   # grid over ability
theta_vals = np.linspace(0.01, 0.99, 20) # threshold values
f_k = 10     # concentration of feasibility prior
alpha = 10   # softmax param

# CONTEXTS: single parameter f_ctx (common-ground feasibility)
contexts = {
    'benchpress 150': 0.10,
    'drive me home':  0.70,
    'call me later':  0.90,
    'pass the salt':  0.99,
}

In [ ]:
## Primitives

# Goal prior: action goals presuppose feasibility.
# Given feasibility, max-entropy → P(action) = 0.5 * f_ctx
@jax.jit
def goal_prior(g, f_ctx):
    p_action = 0.5 * f_ctx
    return jnp.where(g == ACTION, p_action, 1 - p_action)

# Feasibility prior: Beta prior over mu, concentrated around f_ctx
def f_prior(mu, f_ctx):
    mu_c = jnp.clip(mu, .01, .99)
    return jax_beta.pdf(mu_c, f_ctx * f_k, (1 - f_ctx) * f_k)

# Information gain: MI between mu and the answer to "can you?" (mu > t)
@jax.jit
def discrete_entropy(probs):
    p_c = jnp.clip(probs, 1e-12, 1.0)
    return -jnp.sum(p_c * jnp.log(p_c))

@jax.jit
def info_gain(f_ctx, t):
    mu_arr = jnp.array(mu_vals)
    raw = jax_beta.pdf(jnp.clip(mu_arr, .01, .99), f_ctx * f_k, (1 - f_ctx) * f_k)
    pi_f = raw / jnp.sum(raw)
    p_y = jnp.where(mu_arr > t, 1.0, 0.0)
    p_yes_marg = jnp.sum(pi_f * p_y)
    p_no_marg = 1.0 - p_yes_marg
    post_yes = pi_f * p_y / jnp.clip(p_yes_marg, 1e-12)
    post_no = pi_f * (1 - p_y) / jnp.clip(p_no_marg, 1e-12)
    return discrete_entropy(pi_f) - p_yes_marg * discrete_entropy(post_yes) \
                                  - p_no_marg * discrete_entropy(post_no)

# Response utility: ACT entails YES (acting answers the ability question)
@jax.jit
def eu_respond(r, g, mu, t):
    can = jnp.where(mu > t, 1.0, 0.0)
    u_act = can
    u_pass = jnp.where(g == ACTION, 0., 1.)
    return jnp.where(r == ACT, u_act, u_pass)

print("IG by context (avg over theta):")
for label, f_ctx in contexts.items():
    igs = [float(info_gain(f_ctx, t)) for t in theta_vals]
    print(f"  {label:<18} f_ctx={f_ctx:.2f}  avg_IG={np.mean(igs):.4f}")

In [ ]:
## S1
u_label = {CAN: 'can', IMP: 'imp', NULL: 'null'}
goal_label = {INFO: 'info', ACTION: 'action'}

@jax.jit
def eu_s1(u, g, mu, t, f_ctx):
    comply = jnp.where(mu > t, 1.0, 0.0)
    act = jnp.where(u == IMP, comply, 0.)
    info = jnp.where(u == CAN, info_gain(f_ctx, t), 0.)
    social = jnp.where(u == IMP, c_imp, 0.)
    question = jnp.where(u == CAN, c_can, 0.)
    return jnp.where(g == ACTION, act, info) - social - question

@memo
def S1[_u: U, _g: goals, _mu: mu_vals, _t: theta_vals](f_ctx, alpha):
    speaker: knows(_g, _mu, _t)
    speaker: chooses(u in U, wpp=exp(alpha * eu_s1(u, _g, _mu, _t, f_ctx)))
    return Pr[speaker.u == _u]

s1_out = S1(f_ctx=0.5, alpha=alpha)
ti_mid = int(len(theta_vals) // 2)
for gi, g in enumerate(goals):
    probs = {u_label[u]: round(float(s1_out[ui, gi, 25, ti_mid]), 3) for ui, u in enumerate(U)}
    print(f"S1(g={goal_label[g]}, mu=.5, t={theta_vals[ti_mid]:.2f}) = {probs}")

In [ ]:
## A1 — pointwise rational responder
# A1 knows (mu, t) and observes u=CAN.
# Uses wants/EU to jointly infer goal and choose response.
# ACT entails YES: acting answers the ability question.

@memo
def A1[_r: R, _mu: mu_vals, _t: theta_vals](f_ctx, alpha):
    a1: knows(_mu, _t)
    a1: thinks[
        spk: knows(_mu, _t),
        spk: given(g in goals, wpp=goal_prior(g, f_ctx)),
        spk: chooses(u in U, wpp=exp(alpha * eu_s1(u, g, _mu, _t, f_ctx)))
    ]
    a1: observes_that[spk.u == 0]
    a1: wants(payoff = eu_respond(r, spk.g, _mu, _t))
    a1: chooses(r in R, wpp=exp(alpha * EU[payoff]))
    return Pr[a1.r == _r]

@memo
def A1_goal[_g: goals, _mu: mu_vals, _t: theta_vals](f_ctx, alpha):
    a1: knows(_g, _mu, _t)
    a1: thinks[
        spk: knows(_mu, _t),
        spk: given(g in goals, wpp=goal_prior(g, f_ctx)),
        spk: chooses(u in U, wpp=exp(alpha * eu_s1(u, g, _mu, _t, f_ctx)))
    ]
    a1: observes_that[spk.u == 0]
    return a1[Pr[spk.g == _g]]

def marginalize(table, idx, f_ctx):
    """Marginalize table[idx] over mu (prior-weighted) and theta (uniform)."""
    mu_arr = jnp.array(mu_vals)
    raw = jax_beta.pdf(jnp.clip(mu_arr, .01, .99), f_ctx * f_k, (1 - f_ctx) * f_k)
    w_f = raw / jnp.sum(raw)
    return float(jnp.mean(jnp.sum(w_f[:, None] * table[idx], axis=0)))

_ = A1(f_ctx=0.5, alpha=alpha)
print("A1 by context:")
for label, f_ctx in contexts.items():
    g1 = marginalize(A1_goal(f_ctx=f_ctx, alpha=alpha), ACTION, f_ctx)
    p_a1 = marginalize(A1(f_ctx=f_ctx, alpha=alpha), ACT, f_ctx)
    print(f"  {label:<18} f_ctx={f_ctx:.2f}  P(g=act)={g1:.3f}  P(act)={p_a1:.3f}")

In [ ]:
## S2
# S2 anticipates A1's response via inline JAX computation

@jax.jit
def a1_act_inline(mu, t, f_ctx):
    """A1's P(ACT | mu, t) as pure JAX — for S2's expected compliance."""
    eu_act_g = jnp.array([eu_s1(u, ACTION, mu, t, f_ctx) for u in U])
    eu_inf_g = jnp.array([eu_s1(u, INFO, mu, t, f_ctx) for u in U])
    def softmax_at(eus, idx):
        logits = alpha * eus
        logits = logits - logits.max()
        p = jnp.exp(logits)
        return p[idx] / p.sum()
    s1_can_act = softmax_at(eu_act_g, CAN)
    s1_can_inf = softmax_at(eu_inf_g, CAN)
    p_act_prior = 0.5 * f_ctx
    p_ga = p_act_prior * s1_can_act / (p_act_prior * s1_can_act + (1 - p_act_prior) * s1_can_inf + 1e-20)
    can = jnp.where(mu > t, 1.0, 0.0)
    logits = alpha * jnp.array([can, 1.0 - p_ga])
    logits = logits - logits.max()
    p = jnp.exp(logits)
    return p[ACT] / p.sum()

@jax.jit
def eu_s2(u, g, mu, t, f_ctx):
    a1_act = a1_act_inline(mu, t, f_ctx)
    comply = jnp.where(mu > t, 1.0, 0.0)
    act_can = jnp.where(u == CAN, a1_act, 0.)
    act_imp = jnp.where(u == IMP, comply, 0.)
    act = jnp.where(u == CAN, act_can, act_imp)
    info = jnp.where(u == CAN, info_gain(f_ctx, t), 0.)
    social = jnp.where(u == IMP, c_imp, 0.)
    question = jnp.where(u == CAN, c_can, 0.)
    return jnp.where(g == ACTION, act, info) - social - question

@memo
def S2[_u: U, _g: goals, _mu: mu_vals, _t: theta_vals](f_ctx, alpha):
    speaker: knows(_g, _mu, _t)
    speaker: chooses(u in U, wpp=exp(alpha * eu_s2(u, _g, _mu, _t, f_ctx)))
    return Pr[speaker.u == _u]

s2_out = S2(f_ctx=0.5, alpha=alpha)
for gi, g in enumerate(goals):
    probs = {u_label[u]: round(float(s2_out[ui, gi, 25, ti_mid]), 3) for ui, u in enumerate(U)}
    print(f"S2(g={goal_label[g]}, mu=.5, t={theta_vals[ti_mid]:.2f}) = {probs}")

In [ ]:
## A2

@memo
def A2[_r: R, _mu: mu_vals, _t: theta_vals](f_ctx, alpha):
    a2: knows(_mu, _t)
    a2: thinks[
        spk: knows(_mu, _t),
        spk: given(g in goals, wpp=goal_prior(g, f_ctx)),
        spk: chooses(u in U, wpp=exp(alpha * eu_s2(u, g, _mu, _t, f_ctx)))
    ]
    a2: observes_that[spk.u == 0]
    a2: wants(payoff = eu_respond(r, spk.g, _mu, _t))
    a2: chooses(r in R, wpp=exp(alpha * EU[payoff]))
    return Pr[a2.r == _r]

@memo
def A2_goal[_g: goals, _mu: mu_vals, _t: theta_vals](f_ctx, alpha):
    a2: knows(_g, _mu, _t)
    a2: thinks[
        spk: knows(_mu, _t),
        spk: given(g in goals, wpp=goal_prior(g, f_ctx)),
        spk: chooses(u in U, wpp=exp(alpha * eu_s2(u, g, _mu, _t, f_ctx)))
    ]
    a2: observes_that[spk.u == 0]
    return a2[Pr[spk.g == _g]]

_ = A2(f_ctx=0.5, alpha=alpha)
print("A2 by context:")
for label, f_ctx in contexts.items():
    g2 = marginalize(A2_goal(f_ctx=f_ctx, alpha=alpha), ACTION, f_ctx)
    p_a2 = marginalize(A2(f_ctx=f_ctx, alpha=alpha), ACT, f_ctx)
    print(f"  {label:<18} f_ctx={f_ctx:.2f}  P(g=act)={g2:.3f}  P(act)={p_a2:.3f}")

In [ ]:
## plots: A1 vs A2

# Summary table
print(f"{'context':<18} {'f_ctx':>5} {'P(g=a|A1)':>10} {'P(act|A1)':>10} {'P(g=a|A2)':>10} {'P(act|A2)':>10}")
for label, f_ctx in contexts.items():
    g1 = marginalize(A1_goal(f_ctx=f_ctx, alpha=alpha), ACTION, f_ctx)
    p_a1 = marginalize(A1(f_ctx=f_ctx, alpha=alpha), ACT, f_ctx)
    g2 = marginalize(A2_goal(f_ctx=f_ctx, alpha=alpha), ACTION, f_ctx)
    p_a2 = marginalize(A2(f_ctx=f_ctx, alpha=alpha), ACT, f_ctx)
    print(f"  {label:<18} {f_ctx:>5.2f} {g1:>10.3f} {p_a1:>10.3f} {g2:>10.3f} {p_a2:>10.3f}")

# Monotonicity checks
ga1 = [marginalize(A1_goal(f_ctx=f, alpha=alpha), ACTION, f) for f in contexts.values()]
ga2 = [marginalize(A2_goal(f_ctx=f, alpha=alpha), ACTION, f) for f in contexts.values()]
va1 = [marginalize(A1(f_ctx=f, alpha=alpha), ACT, f) for f in contexts.values()]
va2 = [marginalize(A2(f_ctx=f, alpha=alpha), ACT, f) for f in contexts.values()]
print(f"\nA1 goal mono:       {all(a < b for a, b in zip(ga1, ga1[1:]))}")
print(f"A2 goal mono:       {all(a < b for a, b in zip(ga2, ga2[1:]))}")
print(f"A1 compliance mono: {all(a < b for a, b in zip(va1, va1[1:]))}")
print(f"A2 compliance mono: {all(a < b for a, b in zip(va2, va2[1:]))}")

# Sweep over f_ctx
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fs = np.linspace(0.05, 0.99, 40)

ax = axes[0]
a1_goals = [marginalize(A1_goal(f_ctx=f, alpha=alpha), ACTION, f) for f in fs]
a2_goals = [marginalize(A2_goal(f_ctx=f, alpha=alpha), ACTION, f) for f in fs]
ax.plot(fs, a1_goals, label='A1', color='steelblue', linewidth=2)
ax.plot(fs, a2_goals, label='A2', color='tomato', linewidth=2)
for label, f_ctx in contexts.items():
    ax.axvline(f_ctx, color='gray', ls=':', lw=0.7, alpha=0.5)
ax.set_xlabel('f_ctx'); ax.set_ylabel('P(g=action | u=can)')
ax.set_title('Goal inference'); ax.legend(); ax.set_ylim(0, 1)

ax = axes[1]
a1_acts = [marginalize(A1(f_ctx=f, alpha=alpha), ACT, f) for f in fs]
a2_acts = [marginalize(A2(f_ctx=f, alpha=alpha), ACT, f) for f in fs]
ax.plot(fs, a1_acts, label='A1', color='steelblue', linewidth=2)
ax.plot(fs, a2_acts, label='A2', color='tomato', linewidth=2)
for label, f_ctx in contexts.items():
    ax.axvline(f_ctx, color='gray', ls=':', lw=0.7, alpha=0.5)
ax.set_xlabel('f_ctx'); ax.set_ylabel('P(act | u=can)')
ax.set_title('Compliance'); ax.legend(); ax.set_ylim(0, 1)

ax = axes[2]
colors_ig = {'benchpress 150': 'steelblue', 'drive me home': 'orange',
             'call me later': 'green', 'pass the salt': 'tomato'}
for label, f_ctx in contexts.items():
    igs = [float(info_gain(f_ctx, t)) for t in theta_vals]
    ax.plot(theta_vals, igs, label=f'{label} (f={f_ctx})', linewidth=2, color=colors_ig[label])
ax.set_xlabel('θ (threshold)'); ax.set_ylabel('Information gain (nats)')
ax.set_title('IG by context'); ax.legend(fontsize=8)

plt.suptitle(f'α={alpha}, c_imp={c_imp}, c_can={c_can}', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
## plots: S1 vs S2
u_colors = {CAN: 'steelblue', IMP: 'tomato', NULL: 'gray'}
u_labels = {CAN: 'can', IMP: 'imp', NULL: 'null'}

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)

for ci, (ctx_name, f_ctx) in enumerate([('benchpress (f=.10)', 0.10), ('salt (f=.99)', 0.99)]):
    s1_ctx = S1(f_ctx=f_ctx, alpha=alpha)
    s2_ctx = S2(f_ctx=f_ctx, alpha=alpha)

    for ri, (gi, glabel) in enumerate([(ACTION, 'g=action'), (INFO, 'g=info')]):
        for si, (out, mlabel) in enumerate([(s1_ctx, 'S1'), (s2_ctx, 'S2')]):
            ax = axes[ri, ci * 2 + si]
            for ui in U:
                ax.plot(mu_vals, out[ui, gi, :, ti_mid],
                        color=u_colors[ui], label=u_labels[ui], linewidth=2)
            ax.set_title(f'{mlabel} | {glabel}\n{ctx_name}', fontsize=9)
            ax.set_ylim(0, 1)
            if ri == 1: ax.set_xlabel('μ (ability)')
            if ci == 0 and si == 0: ax.set_ylabel('P(utterance | g, μ)')
            ax.legend(fontsize=8)

plt.suptitle(f'S1 vs S2 at θ={theta_vals[ti_mid]:.2f} — α={alpha}, c_imp={c_imp}', fontsize=12)
plt.tight_layout()
plt.show()